### Mount Drive ###

In [1]:
from google.colab import drive
import os

DRIVE_PATH = '/content/drive'
drive.mount(DRIVE_PATH)
MYDRIVE_PATH = os.path.join(DRIVE_PATH, 'MyDrive/')
TARGET_DIR = os.path.join(MYDRIVE_PATH, 'dsfs-ft-31', 'alzheimer_data/')
TRAIN_AUDIOS_WAV_DIR = os.path.join(TARGET_DIR , 'train_audios','wav/')
print(f'TRAIN_AUDIOS_WAV_DIR : {TRAIN_AUDIOS_WAV_DIR}')
TEST_AUDIOS_WAV_DIR = os.path.join(TARGET_DIR, 'test_audios','wav/')
LABELS_DIR = os.path.join(TARGET_DIR, 'labels/')
TRAIN_LABELS_FILE_PATH = LABELS_DIR + 'train_labels.csv'
print(f'TRAIN_LABELS_FILE_PATH : {TRAIN_LABELS_FILE_PATH}')
TEST_LABELS_FILE_PATH = LABELS_DIR + 'test_labels.csv'

Mounted at /content/drive
TRAIN_AUDIOS_WAV_DIR : /content/drive/MyDrive/dsfs-ft-31/alzheimer_data/train_audios/wav/
TRAIN_LABELS_FILE_PATH : /content/drive/MyDrive/dsfs-ft-31/alzheimer_data/labels/train_labels.csv


In [2]:
%pip install -q mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 155.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.9/753.9 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 25.3 MB/s eta 0:00:00


In [3]:
%pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 142.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 10.0 MB/s eta 0:00:00


### Imports ###

In [4]:
import os
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import librosa


# Deep Learning
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.image import resize
from tensorflow.keras.models import load_model
from tensorflow.keras.losses import CategoricalCrossentropy

# Data Visualization
import librosa.display
import matplotlib.pyplot as plt

# Image Processing
from skimage.transform import resize

# Mlflow
import mlflow
from mlflow.models.signature import infer_signature

import boto3

### Set Mlflow HF space environment variables (to log pipeline training) ###

In [5]:
from getpass import getpass

os.environ["AWS_ACCESS_KEY_ID"] = getpass("AWS Access Key: ")
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("AWS Secret Key: ")
os.environ["MLFLOW_TRACKING_URI"] = getpass("MLFLOW Tracking URI: ")
os.environ["AWS_DEFAULT_REGION"] = "eu-west-3"  # ou autre région

AWS Access Key: ··········
AWS Secret Key: ··········
MLFLOW Tracking URI: ··········


### Initialize Mlflow ###

In [6]:
from mlflow.tracking import MlflowClient

EXPERIMENT_NAME = "VGGNet_30_Reg_Alzheimer"

# End of mlflow previous session
mlflow.end_run()

from mlflow.tracking import MlflowClient

client = MlflowClient()

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

print(experiment)

<Experiment: artifact_location='s3://pfe-ml-bucket/ml-deploy/13', creation_time=1762763146318, experiment_id='13', last_update_time=1762763146318, lifecycle_stage='active', name='VGGNet_30_Reg_Alzheimer', tags={}>


### Define constants and load labels ###

In [7]:
labels_file = TRAIN_LABELS_FILE_PATH  # Path to CSV file
audio_dir = TRAIN_AUDIOS_WAV_DIR    # Path to audio files

# Load labels
labels_df = pd.read_csv(labels_file)

# Get classes (diagnosis_control, diagnosis_mci, diagnosis_adrd)
CLASSES_NAMES = ['diagnosis_control', 'diagnosis_mci', 'diagnosis_adrd']

### Check ###

In [8]:
print(type(labels_df))
print(labels_df.shape)
print(labels_df.columns)
print(labels_df.head())

print("Train label distribution:")
print(labels_df["diagnosis_control"].value_counts(normalize=True))
print(labels_df["diagnosis_mci"].value_counts(normalize=True))
print(labels_df["diagnosis_adrd"].value_counts(normalize=True))

<class 'pandas.core.frame.DataFrame'>
(1646, 4)
Index(['uid', 'diagnosis_control', 'diagnosis_mci', 'diagnosis_adrd'], dtype='object')
    uid  diagnosis_control  diagnosis_mci  diagnosis_adrd
0  aaop                0.0            1.0             0.0
1  abgk                1.0            0.0             0.0
2  ablf                1.0            0.0             0.0
3  acad                1.0            0.0             0.0
4  acis                0.0            1.0             0.0
Train label distribution:
diagnosis_control
1.0    0.553463
0.0    0.446537
Name: proportion, dtype: float64
diagnosis_mci
0.0    0.868165
1.0    0.131835
Name: proportion, dtype: float64
diagnosis_adrd
0.0    0.685298
1.0    0.314702
Name: proportion, dtype: float64


### Generate one test audio file path for pipeline signature generation ###

In [9]:
import shutil

print(f'TRAIN_LABELS_FILE_PATH : {TEST_AUDIOS_WAV_DIR}')
TEST_AUDIOS_WAV_DIR
test_files = [os.path.join(TEST_AUDIOS_WAV_DIR, f) for f in os.listdir(TEST_AUDIOS_WAV_DIR) if os.path.isfile(os.path.join(TEST_AUDIOS_WAV_DIR, f))]
TEST_FILE_PATH = ''
if test_files:
    TEST_FILE_PATH = test_files[0]
else:
    print(f'no files in folder : {TEST_AUDIOS_WAV_DIR}')
print(f'test sample file path : {TEST_FILE_PATH}')

tmp_audio_path = "/tmp/echocare_test.wav"

shutil.copy(TEST_FILE_PATH, tmp_audio_path)

print(f"file copied to {tmp_audio_path} (exist={os.path.exists(tmp_audio_path)})")

TEST_FILE_PATH  = tmp_audio_path

TRAIN_LABELS_FILE_PATH : /content/drive/MyDrive/dsfs-ft-31/alzheimer_data/test_audios/wav/
test sample file path : /content/drive/MyDrive/dsfs-ft-31/alzheimer_data/test_audios/wav/fixv.wav
file copied to /tmp/echocare_test.wav (exist=True)


### Define full pipeline ###
- preprocessing
- model
- model training

In [ ]:
import os
import time
import random
import numpy as np
import librosa
from skimage.transform import resize
import mlflow
import mlflow.keras
from mlflow.tracking import MlflowClient
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

class Alzheimer_VGGNET_Pipeline:
    """
    Pipeline audio pour preprocessing + modèle.
    """
    def __init__(self,
                 sampling_rate=16000,
                 target_shape=(96, 1366),
                 target_duration=30.0,
                 min_duration=25.0,
                 top_threshold_db=30.0,
                 n_mels=128,
                 n_fft=1024,
                 hop_length=256,
                 freq_min=20,
                 freq_max=8000,
                 apply_audio_augmentation=True):

        # Constantes
        self.SAMPLING_RATE = sampling_rate
        self.TARGET_SHAPE = target_shape
        self.TARGET_DURATION = target_duration
        self.MIN_DURATION = min_duration
        self.TOP_THRESHOLD_DB = top_threshold_db
        self.N_MELS = n_mels
        self.N_FFT = n_fft
        self.HOP_LENGTH = hop_length
        self.FREQ_MIN = freq_min
        self.FREQ_MAX = freq_max
        self.APPLY_AUDIO_AUGMENTATION = apply_audio_augmentation

        self.model = None
        self.EXPERIMENT_NAME = EXPERIMENT_NAME

    # Audio files  Preprocessing methods
    def trim_silence(self, audio_data):
        '''
        Remove silences (zones with very low energy) at the beginning and at the end of array
        '''
        if audio_data is None or len(audio_data) == 0:
            return audio_data
        audio_data_trimmed, _ = librosa.effects.trim(audio_data, top_db=self.TOP_THRESHOLD_DB)
        return audio_data_trimmed

    def normalize_audio_by_max(self, audio_data):
        '''
        Normalize audio data so max equals 1 - normalize volume differences between audio / patients
        '''
        if audio_data is None or len(audio_data) == 0:
            return audio_data
        max_val = np.max(np.abs(audio_data))
        if max_val < 1e-6:
            return audio_data
        return audio_data / max_val

    def pad_audio(self, audio_data):
        """
        Add silence at the end of the audio file if its duration is less than target_duration.
        Does not cut the audio if its duration is greater than target_duration.
        """
        target_length = int(self.TARGET_DURATION * self.SAMPLING_RATE)
        if len(audio_data) < target_length:
            padding_length = target_length - len(audio_data)
            audio_data = np.pad(audio_data, (0, padding_length), mode='constant')
        return audio_data

    def augment_audio(self, audio_data):
        """
        Apply random data augmentations to audio signal.
        """
        # Slight time shifting
        # Simulate the fact that an audio recording starts before or after.
        if random.random() < 0.3:
            shift = int(random.uniform(-0.1, 0.1) * len(audio_data))
            audio_data = np.roll(audio_data, shift)
        if random.random() < 0.3:
            n_steps = random.uniform(-1.0, 1.0)
            audio_data = librosa.effects.pitch_shift(audio_data, sr=self.SAMPLING_RATE, n_steps=n_steps)
        if random.random() < 0.3:
            #rate = random.uniform(0.9, 1.1)
            #audio_data = librosa.effects.time_stretch(audio_data, rate=rate)

            rate = random.uniform(0.9, 1.1)
            new_length = int(len(audio_data) / rate)
            audio_data = librosa.resample(y=audio_data, orig_sr=self.SAMPLING_RATE, target_sr=int(self.SAMPLING_RATE * rate))
        if random.random() < 0.3:
            noise_amp = 0.005 * np.random.uniform() * np.amax(audio_data)
            audio_data = audio_data + noise_amp * np.random.normal(size=audio_data.shape)
        return audio_data

    def extract_mel_spectrogram(self, audio_data):
        """
        Extract mel-spectrogram features from an audio signal.
        temporal domain to time-frequency domain
        """
        mel_spec = librosa.feature.melspectrogram(
            y=audio_data,
            sr=self.SAMPLING_RATE,
            n_fft=self.N_FFT,
            hop_length=self.HOP_LENGTH,
            n_mels=self.N_MELS,
            fmin=self.FREQ_MIN,
            fmax=self.FREQ_MAX
        )
        mel_db = librosa.power_to_db(mel_spec, ref=np.max)
        mel_db -= mel_db.min()
        mel_db /= mel_db.max() + 1e-6
        return mel_db

    def resize_spectrogram(self, mel_db):
        """
        Redims mel-spectogram to target_shape
        """
        mel_resized = resize(mel_db, self.TARGET_SHAPE, mode='reflect', anti_aliasing=True)
        mel_resized = np.expand_dims(mel_resized, axis=-1)  # add channel
        return mel_resized

    def preprocess_audio_file(self, file_path, apply_augmentation=False):
        audio_data, sr = librosa.load(file_path, sr=self.SAMPLING_RATE)
        audio_data = self.trim_silence(audio_data)
        audio_data = self.normalize_audio_by_max(audio_data)

        duration_original = librosa.get_duration(y=audio_data, sr=sr)
        if duration_original < self.MIN_DURATION:
            return None

        if apply_augmentation and random.random() < 0.5:
            audio_data = self.augment_audio(audio_data)

        audio_data = self.pad_audio(audio_data)
        mel_spec = self.extract_mel_spectrogram(audio_data)
        mel_spec = self.resize_spectrogram(mel_spec)
        return mel_spec

    def load_dataset(self, audio_dir, labels_df, apply_augmentation=False):
        X, y = [], []
        files_processed_count = 0
        labels_df_size = len(labels_df)
        for _, row in labels_df.iterrows():
            file_path = os.path.join(audio_dir, f"{row['uid']}.wav")
            if not os.path.exists(file_path):
                continue
            mel_spec = self.preprocess_audio_file(file_path, apply_augmentation=apply_augmentation)
            if mel_spec is None:
                continue
            X.append(mel_spec)
            y.append([row['diagnosis_control'], row['diagnosis_mci'], row['diagnosis_adrd']])
            files_processed_count += 1
            if files_processed_count % 100 == 0:
              print(f"Processed: {100*(files_processed_count/labels_df_size):.2f} %")

        return np.array(X), np.array(y)

    # Model (VGGNet with regularization (best model)
    def build_model(self):
        inputs = tf.keras.Input(shape=(*self.TARGET_SHAPE, 1))
        L2 = tf.keras.regularizers.l2(1e-5)
        NUM_CLASSES = len(CLASSES_NAMES)
        x = Conv2D(128, (3, 3), activation='relu', padding='same',kernel_regularizer=L2)(inputs)
        x = MaxPooling2D((2, 4))(x)
        x = Conv2D(256, (3, 3), activation='relu', padding='same',kernel_regularizer=L2)(x)
        x = MaxPooling2D((2, 4))(x)
        x = Conv2D(512, (3, 3), activation='relu', padding='same',kernel_regularizer=L2)(x)
        x = MaxPooling2D((2, 4))(x)
        x = Conv2D(1024, (3, 3), activation='relu', padding='same',kernel_regularizer=L2)(x)
        x = MaxPooling2D((3, 5))(x)
        x = Conv2D(2048, (3, 3), activation='relu', padding='same',kernel_regularizer=L2)(x)
        x = MaxPooling2D((4, 4))(x)
        x = Conv2D(1024, (1,1), activation='relu', padding='same',kernel_regularizer=L2)(x)
        x = Flatten()(x)
        outputs = Dense(NUM_CLASSES, activation='softmax')(x)
        self.model = tf.keras.Model(inputs, outputs)
        self.model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        return self.model

    # log Model training to Mlflow
    def fit(self, X_train, y_train, X_val, y_val, epochs=25, batch_size=32, class_weight=None):
        mlflow.set_experiment(self.EXPERIMENT_NAME)

        with mlflow.start_run():
            mlflow.log_params({
                "epochs": epochs,
                "batch_size": batch_size,
                "sampling_rate": self.SAMPLING_RATE,
                "target_shape": self.TARGET_SHAPE,
                "augmentation": self.APPLY_AUDIO_AUGMENTATION
            })

            history = self.model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=epochs,
                batch_size=batch_size,
                class_weight=class_weight,
                verbose=1
            )

            mlflow.log_metrics({
                "train_acc": history.history["accuracy"][-1],
                "val_acc": history.history["val_accuracy"][-1],
                "train_loss": history.history["loss"][-1],
                "val_loss": history.history["val_loss"][-1]
            })

            # Log pipeline (pre-process + model)
            pipeline_wrapper = Alzheimer_VGGNET_PipelineWrapper(self)

            #input_example = [TEST_FILE_PATH]
            #output_example = pipeline_wrapper.predict(None, input_example)
            #output_example = np.array([[0.3, 0.4, 0.3],[0.1, 0.7, 0.2]])
            #             signature = infer_signature(input_example, output_example)

            signature = infer_signature(
              model_input=[["/fake/path/example.wav"]],
              model_output=np.array([[0.6, 0.1, 0.3]])
            )

            mlflow.pyfunc.log_model(
                artifact_path=EXPERIMENT_NAME.lower() + "_full_pipeline",
                python_model=pipeline_wrapper,
                signature=signature
            )

            return history

    # Predict
    def predict(self, audio_file_path):
        mel_spec = self.preprocess_audio_file(audio_file_path, apply_augmentation=False)
        if mel_spec is None:
            raise ValueError(f"Audio file too short: {audio_file_path}")
        mel_spec = np.expand_dims(mel_spec, axis=0)  # batch dimension
        pred = self.model.predict(mel_spec)
        return pred


### This is a wrapper for pipeline ###

In [ ]:
import mlflow.pyfunc


class Alzheimer_VGGNET_PipelineWrapper(mlflow.pyfunc.PythonModel):
    """
    MLflow wrapper for pipeline preprocessing and model.
    """
    def __init__(self, pipeline):
        self.pipeline = pipeline

    def predict(self, context, model_input):
        """
        model_input : DataFrame or list, with path to audio files
        """
        preds = []

        # Supported formats: DataFrame, list, np.array, etc.
        if isinstance(model_input, pd.DataFrame):
            paths = model_input.iloc[:, 0].tolist()
        elif isinstance(model_input, (list, np.ndarray)):
            paths = model_input
        else:
            raise ValueError(f"entry type not supported : {type(model_input)}")

        flattened = []
        for p in paths:
            if isinstance(p, (list, np.ndarray)):
                flattened.append(p[0])
            else:
                flattened.append(p)

        for audio_path in flattened:
            pred = self.pipeline.predict(audio_path)
            preds.append(pred[0])

        return np.array(preds)

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


### Run audio files pre processing (generating X_train, y_train, X_val, y_val) ###

In [ ]:
pipeline = Alzheimer_VGGNET_Pipeline()
pipeline.build_model()

TEST_PROP = 0.2

labels_df["class_id"] = labels_df[CLASSES_NAMES].idxmax(axis=1)

train_labels_df, val_labels_df = train_test_split(
    labels_df,
    test_size=TEST_PROP,
    random_state=42,
    stratify=labels_df["class_id"]
)

print("Train label distribution:")
print(train_labels_df["class_id"].value_counts(normalize=True))

print("\nValidation label distribution:")
print(val_labels_df["class_id"].value_counts(normalize=True))


X_train, y_train = pipeline.load_dataset(TRAIN_AUDIOS_WAV_DIR, train_labels_df, apply_augmentation=True)
print(f"X Shape : {X_train.shape}")
print(f"y Shape : {y_train.shape}")
counts = np.sum(y_train, axis=0)
for i, c in enumerate(counts):
    print(f"Classe {i} : {int(c)} exemples ({c / np.sum(counts) * 100:.2f}%)")

X_val, y_val = pipeline.load_dataset(TRAIN_AUDIOS_WAV_DIR, val_labels_df, apply_augmentation=False)
counts = np.sum(y_val, axis=0)
for i, c in enumerate(counts):
    print(f"Classe {i} : {int(c)} exemples ({c / np.sum(counts) * 100:.2f}%)")

#pipeline.build_model()
#pipeline.fit(X_train, y_train, X_val, y_val)



Train label distribution:
class_id
diagnosis_control    0.553191
diagnosis_adrd       0.314590
diagnosis_mci        0.132219
Name: proportion, dtype: float64

Validation label distribution:
class_id
diagnosis_control    0.554545
diagnosis_adrd       0.315152
diagnosis_mci        0.130303
Name: proportion, dtype: float64
Processed: 7.60 %
Processed: 15.20 %
Processed: 22.80 %
Processed: 30.40 %
Processed: 37.99 %
Processed: 45.59 %
Processed: 53.19 %
Processed: 60.79 %
Processed: 68.39 %
Processed: 75.99 %
X Shape : (1010, 96, 1366, 1)
y Shape : (1010, 3)
Classe 0 : 555 exemples (54.95%)
Classe 1 : 144 exemples (14.26%)
Classe 2 : 311 exemples (30.79%)
Processed: 30.30 %
Processed: 60.61 %
Classe 0 : 143 exemples (57.66%)
Classe 1 : 30 exemples (12.10%)
Classe 2 : 75 exemples (30.24%)


In [16]:
## Compute class_weight
from sklearn.utils.class_weight import compute_class_weight
import numpy as np


if y_train.ndim > 1:
    y_train_int = np.argmax(y_train, axis=1)
else:
    y_train_int = y_train

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_int),
    y=y_train_int
)

class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

Class weights: {0: np.float64(0.6066066066066066), 1: np.float64(2.337962962962963), 2: np.float64(1.082529474812433)}


### Log train experiment and log pipeline ###

In [17]:
history = pipeline.fit(X_train, y_train, X_val, y_val, epochs=30, batch_size=32)

Epoch 1/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.5193 - loss: 1.6696 - val_accuracy: 0.5766 - val_loss: 1.0795
Epoch 2/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 8s 256ms/step - accuracy: 0.5539 - loss: 1.0453 - val_accuracy: 0.5766 - val_loss: 0.9595
Epoch 3/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 8s 255ms/step - accuracy: 0.5264 - loss: 1.0225 - val_accuracy: 0.5766 - val_loss: 0.9328
Epoch 4/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 8s 257ms/step - accuracy: 0.5598 - loss: 0.9638 - val_accuracy: 0.5766 - val_loss: 0.9352
Epoch 5/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 8s 257ms/step - accuracy: 0.5471 - loss: 0.9677 - val_accuracy: 0.5766 - val_loss: 0.9135
Epoch 6/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 8s 257ms/step - accuracy: 0.5551 - loss: 0.9064 - val_accuracy: 0.5766 - val_loss: 0.9099
Epoch 7/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 8s 258ms/step - accuracy: 0.5572 - loss: 0.9463 - val_accuracy: 0.5766 - val_loss: 0.8953
Epoch 8/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 8s 259ms/step - accuracy: 0.5617 - loss: 0.9372 - val_accuracy: 0.57

2025/11/10 16:46:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/__init__.py:3285: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(


🏃 View run unleashed-shad-166 at: https://pieric-mlflow-server-demo.hf.space/#/experiments/13/runs/6cc5155e7fd941e699a910259a5a995d
🧪 View experiment at: https://pieric-mlflow-server-demo.hf.space/#/experiments/13


### Test invoking pipeline (load pipeline stored above to mlflow experiment/run)###

In [18]:
VGGNET_REG_PIPELINE_URI = "runs:/6cc5155e7fd941e699a910259a5a995d/vggnet_30_reg_alzheimer_full_pipeline"

test_pipeline = mlflow.pyfunc.load_model(VGGNET_REG_PIPELINE_URI)

df = pd.DataFrame([ [ [TEST_FILE_PATH] ] ])
print(df)
print(test_pipeline.predict(df))

                          0
0  [/tmp/echocare_test.wav]
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
[[0.06975073 0.9282327  0.00201657]]
